# Xarray with browser-backed Icechunk I/O

`ipygis` is a bridge to GIS libraries running in the browser. There are a couple of things to know to have it working:

- you must use the `xeus-python` kernel (`ipykernel` currently has a limitation with top-level await and widgets).
- the remote server, or a range-preserving proxy, must allow cross-origin browser requests.
- when using the `@earthmover/icechunk` WASM library, the server must send COOP/COEP headers so that `SharedArrayBuffer` is supported in the browser.

In [ ]:
from ipygis.icechunk import Repository, jupyter_storage
from ipygis.zarr import asynchronous as zarr

In [ ]:
storage = jupyter_storage("examples/hydrosheds.icechunk")
repository = await Repository.open_async(
    storage,
    # backend="@earthmover/icechunk",  # "icechunk-js" is the default
    proxy_url="https://my-proxy.david-brochart.workers.dev/",
    virtual_chunk_prefixes=["https://data.hydrosheds.org/file/hydrosheds-v2/ACC/1s/"],
)
session = await repository.readonly_session_async("main")
session.snapshot_id

In [ ]:
group = await zarr.open_group(session.store, mode="r")
array = await group.getitem("0")
array.shape, array.dtype, array.chunks

In [ ]:
result = await array.getitem((10, 100, 200))
result

In [ ]:
result.item()  # Expected: 7.0

In [ ]:
# Close after finishing all array reads.
await session.aclose()
await repository.aclose()